# 05 — QLoRA de Mistral 7B sur Kaamelott, Platon ou Shakespeare

**Chapitre 3** · durée conseillée : 40 min · **GPU obligatoire** (Colab T4 gratuit suffit) · [![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zenocode-org/ai-masterclass/blob/main/fr/chapitre_3/05_qlora_mistral_colab.ipynb)

![Parcours du chapitre 3](https://raw.githubusercontent.com/zenocode-org/ai-masterclass/main/fr/chapitre_3/figures/00_parcours.png)

On assemble tout : les exemples du notebook 02, LoRA du notebook 03, la quantification NF4 du notebook 04. Le modèle est **Mistral-7B-Instruct-v0.3** (Apache 2.0, 7,25 milliards de paramètres).

> **Avant de commencer (Colab) :** menu *Exécution → Modifier le type d'exécution → GPU T4*. Puis *Exécution → Tout exécuter*. Le téléchargement du modèle (14,5 Go) prend 2 à 4 min.
>
> **Mistral 7B est un modèle de 2024**, retiré du catalogue commercial de Mistral (qui recommande aujourd'hui Ministral 8B). Il reste un excellent objet d'étude : poids ouverts, écosystème LoRA/QLoRA mature, et sa structure est celle du chapitre 2.

**À la fin de ce notebook, vous aurez :**
1. chargé un modèle de 7 milliards de paramètres **en 4 bits** sur un GPU de 15 Go ;
2. regardé ses réponses **avant** fine-tuning ;
3. vérifié que la loss ne porte **que** sur la réponse ;
4. lancé un **QLoRA** avec TRL et lu les courbes train / validation ;
5. comparé les réponses **avant / après** et sauvegardé l'**adapter** ;
6. mesuré le débit (tokens/s), la mémoire maximale et le **coût** d'un entraînement.

In [ ]:
# 0. Installation (Colab) — versions testées ensemble ; ~1 min
import os, sys
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    %pip install -q "transformers==5.22.1" "trl==1.17.1" "peft==0.21.2" "bitsandbytes==0.50.2" "accelerate==1.15.0" "datasets==5.1.0"
    if not os.path.exists("finetune.py"):
        !git clone -q --depth 1 https://github.com/zenocode-org/ai-masterclass
        %cd ai-masterclass/fr/chapitre_3

import math, json, time, random, warnings
warnings.filterwarnings("ignore", message="IProgress not found")
import torch
import matplotlib.pyplot as plt
import finetune as ft

assert torch.cuda.is_available(), "Pas de GPU : sur Colab, Exécution > Modifier le type d'exécution > GPU (T4)."
GPU = torch.cuda.get_device_name(0)
MEMOIRE_GPU = torch.cuda.get_device_properties(0).total_memory / 1024**3
BF16 = torch.cuda.get_device_capability(0)[0] >= 8   # Ampere (A100, L4…) et plus récents ; pas le T4
DTYPE = torch.bfloat16 if BF16 else torch.float16
print(f"GPU : {GPU} ({MEMOIRE_GPU:.1f} Go) | calcul en {'BF16' if BF16 else 'FP16 (pas de BF16 sur ce GPU)'}")
print("PyTorch", torch.__version__)

## 1. Réglages

Changez **`CORPUS`** pour fine-tuner sur un autre texte. Les autres réglages sont des points de départ raisonnables pour un T4 ; chaque choix est justifié dans les notebooks 02 à 04.

In [ ]:
CORPUS = "kaamelott"            # "kaamelott", "platon" ou "shakespeare"
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

# Kaamelott : 38 exemples -> on les prend tous, plusieurs époques.
# Platon / Shakespeare : des milliers d'exemples -> un sous-échantillon, une époque (≈ 15-25 min sur T4).
REGLAGES = {
    "kaamelott":   dict(n_train=None, n_val=None, epoques=3, eval_steps=5),
    "platon":      dict(n_train=800, n_val=80, epoques=1, eval_steps=10),
    "shakespeare": dict(n_train=800, n_val=80, epoques=1, eval_steps=10),
}[CORPUS]

MAX_LENGTH = 512                  # notebook 02 : couvre ~99 % des exemples
LORA_R, LORA_ALPHA = 16, 32       # notebook 04 : 42 M paramètres entraînables
LEARNING_RATE = 2e-4              # ordre de grandeur usuel pour LoRA / QLoRA
BATCH, ACCUMULATION = 4, 4        # batch effectif 16 (notebook 04, §6)
if CORPUS == "kaamelott":
    ACCUMULATION = 2              # 38 exemples : batch effectif 8, sinon trop peu de pas
DOSSIER_ADAPTER = f"adapters/mistral7b-{CORPUS}-qlora"
GRAINE = 42

## 2. Les données du notebook 02

Les fichiers `data/<corpus>_{train,val}.jsonl` ont été produits au notebook 02 (split **par scène / dialogue**). On ne garde que les colonnes `prompt` et `completion` : c'est le format **prompt-completion conversationnel** de TRL, qui calcule la loss **seulement sur la completion**.

In [ ]:
from datasets import Dataset

def charger(split, n):
    exemples = ft.charger_jsonl(f"data/{CORPUS}_{split}.jsonl")
    random.Random(GRAINE).shuffle(exemples)
    exemples = exemples[:n] if n else exemples
    return exemples, Dataset.from_list([{"prompt": e["prompt"], "completion": e["completion"]} for e in exemples])

train_ex, train_ds = charger("train", REGLAGES["n_train"])
val_ex, val_ds = charger("val", REGLAGES["n_val"])
print(f"{CORPUS} : {len(train_ds)} exemples d'entraînement, {len(val_ds)} de validation")
print(json.dumps(train_ds[0], ensure_ascii=False, indent=1)[:800])

## 3. Charger Mistral 7B en 4 bits

`BitsAndBytesConfig` reprend exactement le notebook 04 : NF4, double quantification, calcul en BF16 (ou FP16 sur T4). La quantification a lieu **pendant le chargement**, couche par couche : le modèle BF16 complet n'est jamais en mémoire GPU.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.unk_token        # Mistral n'a pas de token de padding : on prend <unk>

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=DTYPE,
)
debut = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, dtype=DTYPE, device_map={"": 0},
)
model.config.use_cache = False                    # incompatible avec le gradient checkpointing
print(f"chargé en {time.time() - debut:.0f} s")
print(f"empreinte du modèle : {model.get_memory_footprint() / 1024**3:.2f} Go (BF16 : 14,5 Go)")
print(f"mémoire GPU utilisée : {torch.cuda.memory_allocated() / 1024**3:.2f} Go sur {MEMOIRE_GPU:.1f}")
print(model.model.layers[0].self_attn.q_proj)

`Linear4bit` : chaque `q_proj` stocke ses 16,8 M de poids sur 4 bits. ≈ 4 Go au total, comme prévu au notebook 04.

## 4. Avant le fine-tuning : que répond le modèle ?

On garde **des prompts fixes**, pris dans la **validation** (scènes jamais vues pendant l'entraînement), pour comparer avant / après. Graine fixée : seule la différence de poids change les réponses.

In [ ]:
N_PROMPTS = 4
prompts_fixes = val_ex[:N_PROMPTS]
model.eval()
avant = []
for e in prompts_fixes:
    avant.append(ft.generer(model, tokenizer, e["prompt"], max_new_tokens=80))
    print("CONTEXTE :", e["prompt"][0]["content"].split("\n\n", 1)[1][-300:])
    print("ATTENDU  :", e["completion"][0]["content"])
    print("MISTRAL  :", avant[-1])
    print("-" * 100)

Le modèle Instruct « de base » répond souvent **comme un assistant** : il commente la scène, propose plusieurs répliques, explique le contexte, ou écrit dans un français standard. Il connaît peut-être Kaamelott ou Platon, mais il n'a pas été entraîné à **continuer ce format** dans ce registre.

## 5. Ajouter les adapters LoRA

Exactement le `ft.appliquer_lora` du notebook 03, version `peft` : on gèle tout, on enveloppe les 7 types de couches linéaires des 32 blocs.

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("calcul du notebook 04 :", f"{ft.parametres_lora(LORA_R, lora_config.target_modules):,}")
print(model.base_model.model.model.layers[0].self_attn.q_proj)

## 6. Configurer l'entraînement (TRL)

`SFTTrainer` (bibliothèque **TRL** de Hugging Face) fait la boucle du chapitre 2 — forward, loss, backward, step, évaluation — avec tout ce qu'on a vu :

| Argument | Valeur | Vu dans |
|---|---|---|
| `max_length` | 512 | notebook 02, distribution des longueurs |
| `completion_only_loss` | `True` | notebook 01, masque `-100` |
| `per_device_train_batch_size` × `gradient_accumulation_steps` | 4 × 4 = 16 | notebook 04, accumulation |
| `gradient_checkpointing` | `True` | notebook 04, activations |
| `learning_rate`, `lr_scheduler_type`, `warmup_steps` | 2e-4, cosinus, 10 % | lr plus grand pour LoRA (notebook 03) ; montée progressive puis descente |
| `optim` | `paged_adamw_8bit` | notebook 04, optimiseurs paginés |
| `bf16` / `fp16` | selon le GPU | T4 : FP16 |

In [ ]:
from trl import SFTConfig, SFTTrainer

args = SFTConfig(
    output_dir=f"checkpoints/{CORPUS}",
    num_train_epochs=REGLAGES["epoques"],
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=0.1,                       # un nombre < 1 = une fraction des pas
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=ACCUMULATION,
    max_length=MAX_LENGTH,
    completion_only_loss=True,
    packing=False,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    bf16=BF16, fp16=not BF16,
    logging_steps=1,
    eval_strategy="steps", eval_steps=REGLAGES["eval_steps"],
    save_strategy="no",                     # on sauvegarde l'adapter à la main à la fin
    seed=GRAINE, data_seed=GRAINE,
    report_to="none",
)
trainer = SFTTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, processing_class=tokenizer)

if not BF16:
    # TRL convertit les adapters en BF16 pour un modèle 4 bits ; le T4 ne gère pas le BF16 :
    # on les garde en FP32 (le calcul se fait en FP16 grâce à l'autocast).
    for p in trainer.model.parameters():
        if p.requires_grad:
            p.data = p.data.float()

n_pas = math.ceil(len(train_ds) / (BATCH * ACCUMULATION)) * REGLAGES["epoques"]
print(f"{n_pas} pas d'optimisation prévus (batch effectif {BATCH * ACCUMULATION})")

### Vérifier le masque avant de lancer

Une erreur classique : entraîner sur le prompt **et** la réponse (le modèle apprend alors à recopier des consignes). On prend un batch tel que le *trainer* le fabrique et on décode les tokens dont le label n'est **pas** `-100` :

In [ ]:
batch = next(iter(trainer.get_train_dataloader()))
ids, labels = batch["input_ids"][0], batch["labels"][0]
print("forme du batch :", tuple(batch["input_ids"].shape))
print(f"tokens dans la séquence : {int(batch['attention_mask'][0].sum())}, dont {int((labels != -100).sum())} comptent dans la loss")
print("\nCE QUI COMPTE DANS LA LOSS :", repr(tokenizer.decode(ids[labels != -100])))
print("\nCE QUI EST MASQUÉ (début)  :", repr(tokenizer.decode(ids[(labels == -100) & (batch["attention_mask"][0] == 1)])[:300]))

Seule la réplique (et `</s>`) compte.

## 7. Entraîner

La loss de validation **avant** entraînement sert de point de départ (au départ `B = 0` : c'est exactement le modèle de base). Puis on lance. Sur un T4 : quelques minutes pour Kaamelott, 15 à 25 min pour 800 exemples de Platon ou Shakespeare.

In [ ]:
torch.cuda.reset_peak_memory_stats()
eval_avant = trainer.evaluate()["eval_loss"]
print(f"loss de validation avant fine-tuning : {eval_avant:.3f} (perplexité {math.exp(eval_avant):.1f})")

debut = time.time()
resultat = trainer.train()
duree = time.time() - debut
eval_apres = trainer.evaluate()["eval_loss"]
print(f"\nentraînement : {duree / 60:.1f} min | loss de validation : {eval_avant:.3f} -> {eval_apres:.3f} "
      f"(perplexité {math.exp(eval_avant):.1f} -> {math.exp(eval_apres):.1f})")
print(f"mémoire GPU maximale : {torch.cuda.max_memory_allocated() / 1024**3:.1f} Go sur {MEMOIRE_GPU:.1f}")

In [ ]:
hist = trainer.state.log_history
pas_train = [h["step"] for h in hist if "loss" in h]
loss_train = [h["loss"] for h in hist if "loss" in h]
pas_val = [0] + [h["step"] for h in hist if "eval_loss" in h][1:-1]
loss_val = [eval_avant] + [h["eval_loss"] for h in hist if "eval_loss" in h][1:-1]
if pas_val[-1] != trainer.state.global_step:
    pas_val.append(trainer.state.global_step); loss_val.append(eval_apres)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(pas_train, loss_train, alpha=0.6, label="entraînement (par pas)")
ax.plot(pas_val, loss_val, marker="o", lw=2, label="validation")
ax.set_xlabel("pas d'optimisation"); ax.set_ylabel("loss (tokens de réponse)"); ax.legend()
ax.set_title(f"QLoRA Mistral 7B — {CORPUS}")
plt.show()

**Lire la courbe** (même grille qu'au chapitre 2). Ce que l'on s'attend à voir :
- la loss de **validation** baisse nettement dès les premiers pas : le modèle apprend le **format** et le **registre**, ce qui est facile pour un modèle qui sait déjà parler ;
- la loss d'**entraînement** est bruitée (un point = un batch de 8 ou 16 exemples) ;
- sur **Kaamelott**, la loss d'entraînement continue de baisser alors que la validation stagne ou remonte à la 3ᵉ époque : **mémorisation** de 38 exemples (notebook 06) ;
- sur **Platon / Shakespeare**, on a vu chaque exemple une seule fois : train et validation restent proches.

## 8. Après le fine-tuning : mêmes prompts, même graine

In [ ]:
model.eval()
model.config.use_cache = True
for e, rep_avant in zip(prompts_fixes, avant):
    print("CONTEXTE :", e["prompt"][0]["content"].split("\n\n", 1)[1][-300:])
    print("ATTENDU  :", e["completion"][0]["content"])
    print("AVANT    :", rep_avant)
    print("APRÈS    :", ft.generer(model, tokenizer, e["prompt"], max_new_tokens=80))
    print("-" * 100)

Ce qui doit changer : **une seule réplique**, au format `NOM: texte`, dans le **registre** du corpus, puis `</s>` (le modèle s'arrête). Ce qui ne doit pas vous tromper : une réponse « parfaite » peut être une **citation apprise par cœur** — c'est l'objet du notebook 06.

### Une scène inventée

Le vrai test : un contexte qui n'existe nulle part dans le corpus.

In [ ]:
SCENES_INVENTEES = {
    "kaamelott": "ARTHUR: Bon. Quelqu’un a une idée pour récupérer le Graal avant ce soir ?\nPERCEVAL: Moi j’ai une idée, mais elle est pas finie.",
    "platon": "SOCRATE: Dis-moi, Phèdre, crois-tu qu'une machine qui parle sait ce qu'elle dit ?\nPHÈDRE: Je ne sais, Socrate ; il me semble qu'elle parle fort bien.",
    "shakespeare": "HAMLET:\nWhat is this engine that doth speak like men?\nHORATIO:\nA thing of numbers, lord, and borrowed words.",
}
prompt = [{"role": "user", "content": f"{ft.CONSIGNES[CORPUS]}\n\n{SCENES_INVENTEES[CORPUS]}"}]
for graine in range(3):
    print(f"[{graine}]", ft.generer(model, tokenizer, prompt, max_new_tokens=80, temperature=0.8, graine=graine))

## 9. Débit, coût, sauvegarde

Le coût d'un entraînement se prévoit avec le **débit** mesuré, pas en devinant (notebook 02) :

$$\text{GPU-heures} \approx \frac{N_\text{tokens} \times N_\text{époques}}{\text{débit (tokens/s)} \times 3600}, \qquad \text{coût} = \text{GPU-heures} \times \text{prix horaire}$$

In [ ]:
def n_tokens(e):
    return min(MAX_LENGTH, len(tokenizer.apply_chat_template(e["prompt"] + e["completion"])["input_ids"]))

tokens_vus = sum(n_tokens(e) for e in train_ex) * REGLAGES["epoques"]
debit = tokens_vus / resultat.metrics["train_runtime"]
print(f"{tokens_vus:,} tokens d'entraînement en {resultat.metrics['train_runtime']:.0f} s -> {debit:,.0f} tokens/s sur {GPU}")

tous = ft.charger_jsonl(f"data/{CORPUS}_train.jsonl")
tokens_corpus = sum(n_tokens(e) for e in tous[:2000]) / min(2000, len(tous)) * len(tous)
heures = tokens_corpus / debit / 3600
print(f"\nUne époque sur TOUT {CORPUS} ({len(tous):,} exemples, ~{tokens_corpus:,.0f} tokens) : ~{heures:.1f} GPU-heure(s) sur ce GPU")

# Prix de location à l'heure, ordres de grandeur (nov. 2026) : à vérifier chez le fournisseur le jour J.
PRIX_HEURE = {"T4": 0.35, "L4": 0.70, "A10": 1.00, "A100": 1.80, "H100": 2.50}
prix = next((p for nom, p in PRIX_HEURE.items() if nom in GPU), None)
if prix:
    print(f"coût en location : ~{heures:.1f} h x {prix:.2f} $/h = ~{heures * prix:.2f} $ (gratuit sur Colab, dans la limite des quotas)")

On sauvegarde **seulement l'adapter** (les matrices A et B), comme au notebook 03 :

In [ ]:
from pathlib import Path
trainer.save_model(DOSSIER_ADAPTER)
tokenizer.save_pretrained(DOSSIER_ADAPTER)
for f in sorted(Path(DOSSIER_ADAPTER).iterdir()):
    print(f"{f.name:32s} {f.stat().st_size / 1024**2:8.1f} Mo")
print(json.load(open(f"{DOSSIER_ADAPTER}/adapter_config.json"))["base_model_name_or_path"])

≈ 160 Mo d'adapter contre 14,5 Go de modèle. Le fichier `adapter_config.json` retient le **modèle de base** : l'adapter seul est inutile sans lui.

**Le disque de Colab est effacé** à la fin de la session. Pour garder l'adapter et le réutiliser au notebook 06, copiez-le sur votre Google Drive (cellule suivante) ou téléchargez-le.

In [ ]:
SAUVER_SUR_DRIVE = False          # passez à True pour copier l'adapter sur Google Drive

if EN_COLAB and SAUVER_SUR_DRIVE:
    import shutil
    from google.colab import drive
    drive.mount("/content/drive")
    destination = f"/content/drive/MyDrive/ai-masterclass/{DOSSIER_ADAPTER}"
    shutil.copytree(DOSSIER_ADAPTER, destination, dirs_exist_ok=True)
    print("copié dans", destination)
# Autre option : téléchargement d'une archive
# import shutil; from google.colab import files
# files.download(shutil.make_archive(DOSSIER_ADAPTER, "zip", DOSSIER_ADAPTER))

## À retenir

- QLoRA = `BitsAndBytesConfig` (base NF4 gelée) + `LoraConfig` (adapters) + `SFTTrainer` (la boucle du chapitre 2). Mistral 7B tient en ~4 Go et s'entraîne sur un T4.
- **Vérifier le masque** avant de lancer : seuls les tokens de la réponse comptent.
- La loss de validation baisse vite : le modèle apprend d'abord le **format** et le **registre**.
- Le coût se prévoit avec le **débit mesuré** (tokens/s) : on lance un petit run, on mesure, on extrapole.
- On sauvegarde l'**adapter** (~160 Mo), pas le modèle.

**Pannes fréquentes**

| Symptôme | Causes probables | Que faire |
|---|---|---|
| `CUDA out of memory` | séquences trop longues, batch trop grand | réduire `MAX_LENGTH`, puis `BATCH` (et augmenter `ACCUMULATION`), vérifier le chargement 4 bits |
| loss `nan` | FP16 instable, lr trop grand | baisser `LEARNING_RATE`, BF16 si le GPU le permet |
| la loss chute vers 0, les réponses sont des citations | surapprentissage, fuite train/val | moins d'époques, vérifier le split par scène (notebook 02) |
| la loss baisse mais le modèle « parle comme avant » | le masque est faux, ou trop peu de pas | revoir le §6, augmenter les pas |
| le modèle ne s'arrête jamais | `</s>` absent de la cible | vérifier le chat template (notebook 01, §4) |

**Expériences** (une variable à la fois, même graine, même split) : `LORA_R = 8` contre `32` ; `target_modules` limité à l'attention ; 1 contre 3 époques sur Kaamelott ; `CORPUS = "platon"` avec 200 contre 800 exemples.